In [1]:
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForSeq2SeqLM

/home/pipa03/p3-llm/c2-transformer/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
# bert모델 - 인코더만 사용
tokenizer = AutoTokenizer.from_pretrained("klue/bert-base")
body = AutoModel.from_pretrained("klue/bert-base")
body.eval()     # 평가모드

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2347.36it/s]
[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(32000, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
            (dropout): Dropout(p=

In [5]:
body.config.num_hidden_layers   # 설정값 확인 -> 총 층수 12

12

In [10]:
# BART 모델 - 한국어 사전학습 모델을 파인튜닝해서 올려둔 모델
kobart_tokenizer = AutoTokenizer.from_pretrained("gogamza/kobart-base-v2")
kobart = AutoModelForSeq2SeqLM.from_pretrained("gogamza/kobart-summarization")
kobart.eval() 

Loading weights: 100%|██████████| 260/260 [00:00<00:00, 2284.04it/s]


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(30000, 768, padding_idx=3)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(30000, 768, padding_idx=3)
      (embed_positions): BartLearnedPositionalEmbedding(1028, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (fi

In [18]:
# 문장 토큰화
text = "나는 어제 도서관에서 책을 빌렸다."

inputs = tokenizer(text, return_tensors="pt")
tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

['[CLS]',
 '나',
 '##는',
 '어제',
 '도서관',
 '##에서',
 '책',
 '##을',
 '빌렸',
 '##다',
 '.',
 '[SEP]']

In [20]:
with torch.no_grad():
    outputs = body(**inputs)

In [21]:
outputs

BaseModelOutputWithPoolingAndCrossAttentions(last_hidden_state=tensor([[[ 0.2706, -0.5882, -0.2757,  ...,  0.9870, -0.1088,  0.0545],
         [-1.4213, -1.0361, -0.5983,  ...,  1.2247,  0.0362,  0.6989],
         [-0.6815, -1.1524,  0.6063,  ...,  1.1478,  0.8470, -0.8086],
         ...,
         [-0.0066,  0.6564, -1.3396,  ...,  0.6719, -0.6876,  0.4835],
         [ 0.8967, -0.4278,  0.7150,  ...,  0.6001, -0.4296,  0.0611],
         [ 1.0937, -0.2357,  0.6240,  ...,  0.5415, -0.3400,  0.4143]]]), pooler_output=tensor([[ 4.1966e-01, -1.2048e-01,  6.9188e-01,  6.5178e-01,  2.5799e-01,
         -5.5168e-01,  9.7246e-01,  1.9297e-01, -8.5281e-01, -8.9727e-01,
          1.7544e-01, -1.0000e+00, -1.3942e-01, -7.7765e-03,  3.0018e-02,
         -2.3033e-01,  1.9286e-01, -4.5162e-01, -9.5401e-02, -1.4889e-01,
          4.3962e-01,  6.9353e-01,  2.6702e-02,  9.9862e-01,  1.2700e-01,
         -1.8444e-01,  4.3138e-01,  8.0600e-01,  7.5535e-01, -8.7728e-01,
          9.8219e-01,  9.2370e-01, -

In [23]:
outputs.last_hidden_state.shape # (문장의 수, 토큰 수, 임베딩 차원)
# 768 개의 숫자 벡터 -> representation. 모델이 문맥을 읽어 계산한 결과

torch.Size([1, 12, 768])

In [25]:
# 특수 토큰
tokenizer.special_tokens_map
# SEP : 구분자
# CLS : 문장의 맨 앞

{'unk_token': '[UNK]',
 'sep_token': '[SEP]',
 'pad_token': '[PAD]',
 'cls_token': '[CLS]',
 'mask_token': '[MASK]'}

In [30]:
from transformers import AutoModelForMaskedLM

mlm = AutoModelForMaskedLM.from_pretrained("klue/bert-base")
mlm.eval()

Loading weights: 100%|██████████| 202/202 [00:00<00:00, 2353.50it/s]
[transformers] BertForMaskedLM LOAD REPORT from: klue/bert-base
Key                         | Status     |  | 
----------------------------+------------+--+-
cls.seq_relationship.bias   | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 
bert.pooler.dense.bias      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertForMaskedLM(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(32000, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, el

In [31]:
mlm.cls

BertOnlyMLMHead(
  (predictions): BertLMPredictionHead(
    (transform): BertPredictionHeadTransform(
      (dense): Linear(in_features=768, out_features=768, bias=True)
      (transform_act_fn): GELUActivation()
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True, bias=True)
    )
    (decoder): Linear(in_features=768, out_features=32000, bias=True)
  )
)

In [36]:
masked_text = f"오늘 점심으로 {tokenizer.mask_token}을 먹었다."
masked_text
inputs = tokenizer(masked_text, return_tensors="pt")
with torch.no_grad():
    outputs = mlm(**inputs)
outputs.logits.shape


torch.Size([1, 11, 32000])

In [38]:
mask_pos = (inputs["input_ids"][0] == tokenizer.mask_token_id).nonzero().item()
scores = outputs.logits[0, mask_pos] 
top_id = scores.argmax().item()                  # argmax: 가장 높은 점수의 위치(단어 번호)
tokenizer.decode(top_id)                        # argmax: 가장 높은 점수의 위치(단어 번호)

'치킨'

In [39]:
#======================

In [43]:
from transformers import AutoConfig
bert_설정 = body.config
bert_설정 = kobart.config
gpt_설정 = AutoConfig.from_pretrained("skt/kogpt2-base-v2")

In [44]:
bert_설정

BartConfig {
  "activation_dropout": 0.0,
  "activation_function": "gelu",
  "add_bias_logits": false,
  "add_final_layer_norm": false,
  "architectures": [
    "BartForConditionalGeneration"
  ],
  "attention_dropout": 0.0,
  "author": "Heewon Jeon(madjakarta@gmail.com)",
  "bos_token_id": 0,
  "classif_dropout": 0.1,
  "classifier_dropout": 0.1,
  "d_model": 768,
  "decoder_attention_heads": 16,
  "decoder_ffn_dim": 3072,
  "decoder_layerdrop": 0.0,
  "decoder_layers": 6,
  "decoder_start_token_id": 2,
  "do_blenderbot_90_layernorm": false,
  "dropout": 0.1,
  "dtype": "float32",
  "encoder_attention_heads": 16,
  "encoder_ffn_dim": 3072,
  "encoder_layerdrop": 0.0,
  "encoder_layers": 6,
  "eos_token_id": 1,
  "extra_pos_embeddings": 2,
  "force_bos_token_to_be_generated": false,
  "forced_eos_token_id": 2,
  "id2label": {
    "0": "NEGATIVE",
    "1": "POSITIVE"
  },
  "init_std": 0.02,
  "is_decoder": false,
  "is_encoder_decoder": true,
  "label2id": {
    "NEGATIVE": 0,
    "POS

In [45]:
print(f"{'':14s} {'BERT':>10s} {'GPT-2':>10s} {'BART':>12s}")
print(f"{'임베딩 차원':12s} {bert_설정.hidden_size:>10d} {gpt_설정.n_embd:>10d} {bart_설정.d_model:>12d}")
print(f"{'헤드 수':12s} {bert_설정.num_attention_heads:>10d} {gpt_설정.n_head:>10d} {bart_설정.encoder_attention_heads:>12d}")
print(f"{'층 수':12s} {bert_설정.num_hidden_layers:>10d} {gpt_설정.n_layer:>10d} {f'{bart_설정.encoder_layers}+{bart_설정.decoder_layers}':>12s}")
print(f"{'단어 수':12s} {bert_설정.vocab_size:>10d} {gpt_설정.vocab_size:>10d} {bart_설정.vocab_size:>12d}")

                     BERT      GPT-2         BART
임베딩 차원              768        768          768
헤드 수                 16         12           16
층 수                   6         12          6+6
단어 수              30000      51200        30000


In [55]:
# bert 모델 한 블럭의 파라미터 수 : 7087872
print (sum( [ p.numel() for p in mlm.bert.encoder.layer[0].parameters() ]))

# bart 모델 한 블럭의 파라미터 수 : 7087872
print (sum( [ p.numel() for p in kobart.model.encoder.layers[0].parameters() ]))

# bart 모델 디코더 한 블럭의 파라미터 수 : 9451776
print (sum( [ p.numel() for p in kobart.model.decoder.layers[0].parameters() ]))


7087872
7087872
9451776


In [56]:
d = gpt_설정.n_embd
d * d * 12 + 13 * d # GPT 모델 인코더 블록 파라미터 수도 이론상 같다.

7087872

In [58]:
viz_model = AutoModel.from_pretrained("klue/bert-base",
                          output_attentions=True,
                          attn_implementation="eager"  )

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2298.69it/s]
[transformers] BertModel LOAD REPORT from: klue/bert-base
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [59]:
viz_model.eval()

배문장들 = [
    "배가 아파서 병원에 갔다.",
    "항구에서 배를 타고 섬으로 갔다.",
    "달고 시원한 배를 깎아 먹었다.",
]

for i, sent in enumerate(배문장들, start=1):
    inputs = tokenizer(sent, return_tensors="pt")
    with torch.no_grad():
        outputs = viz_model(**inputs)

    # outputs.attentions: 층마다 하나씩, 각각 (1, 헤드 12, 토큰 수, 토큰 수) — 어텐션 가중치 전부
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])
    # html_action="return": 화면에 바로 그리는 대신 HTML 문서로 돌려받는다 (파일로 저장하려고)
    viz_html = head_view(outputs.attentions, tokens, html_action="return")

    out_file = Path(f"bertviz_{i}.html")
    out_file.write_text(viz_html.data, encoding="utf-8")
    print(f"[{i}] {sent}  →  {out_file} 저장")

    # 노트북에서 실행 중이면 셀 아래에 바로 띄운다
    if "ipykernel" in sys.modules:
        from IPython.display import display

        display(viz_html)

NameError: name 'head_view' is not defined

In [60]:
# BART 인코더-디코더 (문맥 파악 - 생성)

article = (
    "시립도서관이 다음 달부터 야간 운영을 시작한다. 평일에는 밤 10시까지, 주말에는 밤 8시까지 "
    "열람실과 자료실을 이용할 수 있다. 도서관 측은 직장인과 수험생의 이용 요청이 꾸준히 "
    "늘어난 점을 반영했다고 밝혔다. 야간 시간대에는 사서 두 명이 상주하며, 어린이 자료실은 "
    "기존과 같이 오후 6시에 문을 닫는다. 시는 이용 현황을 본 뒤 다른 구립도서관으로 확대할지 "
    "결정할 계획이다."
)

ids = kobart_tokenizer.encode(article)
input_ids = torch.tensor(
    [[kobart_tokenizer.bos_token_id] + ids + [kobart_tokenizer.eos_token_id]]
)
print(input_ids.shape)  # 입력문 토큰의 길이
summary_ids = kobart.generate(input_ids, max_length=64) # 요약문 토큰 index
print(summary_ids.shape)  # 요약문 토큰의 길이
kobart_tokenizer.decode(summary_ids[0], skip_special_tokens=True) # 요약

torch.Size([1, 87])
torch.Size([1, 32])


'시립도서관이 다음 달부터 야간 운영을 시작하며 평일에는 밤 10시까지, 주말에는 밤 8시까지 열람실과 자료실을 이용할 수 있다.'